# 08: Remediation Experiments

My proposal set a target of macro F1 >= 0.75. Notebook 04's delivered checkpoint reached
0.6754 on the test split (0.6872 under cross-validation, notebook 09). This notebook is my
attempt to close that gap, and a record of what happened when most of it didn't.

| | Experiment | Source |
|---|---|---|
| S0 | Per-class threshold tuning | standard calibration |
| S1 | Loss-function ablation | Lin et al., focal loss |
| S2 | One-vs-rest binary classifiers | Maalej & Nabil, RE'15 |
| S3 | Confident learning label audit | Northcutt et al., JAIR 2021 |
| S4 | Task-adaptive pretraining | Gururangan et al., ACL 2020 |
| S5 | Seed ensembling | Lakshminarayanan et al., NeurIPS 2017 |
| S6 | S2 + S5 combined | (no source) |

Configurations are chosen on validation; test scores are reported but never used to
select.

Runtime is around three hours on Apple Silicon. Run sections individually, not Run All.
Baseline to beat: accuracy 0.7212, macro F1 0.6754.

## Setup

## Note on the stale constants

`BASELINE_F1` above now reads **0.6754**, the delivered checkpoint's test macro F1 after
the focal-loss correction, re-scored from `models/bert_category`. Five-fold
cross-validation (notebook 09) retrains the same configuration from scratch on five
splits and averages 0.6872, which estimates the recipe rather than scoring this
particular checkpoint.

The printed outputs further down were produced with the *previous* constant, **0.7079**,
the pre-correction score. Every "vs baseline" delta below is therefore measured against
0.7079 and reads about 0.03 more negative than it should. For nine of the ten that only
changes the size of the gap, but **S2 crosses zero**: it prints as −0.0311 when against
the corrected baseline it is +0.0014.

A second constant is stale in the same way. The S6 cell compares against **0.7274** as
"S2 alone", a figure from an earlier run again; S2's recorded score is 0.6768. That
printed line reads S6 as 0.015 *worse* than S2, where the recorded scores put it 0.036
*better*.

So two printed comparisons carry the wrong sign: S2 against the baseline, and S6 against
S2. In both cases the raw experiment scores are unaffected — only the constants they were
differenced against changed. Use the scores in
`reports/improvement_experiments_notebook_run.json`, not the printed deltas.

Those outputs are left exactly as they ran rather than regenerated: re-executing this
notebook retrains every model and produces different numbers, which is itself one of
this notebook's findings.

In [1]:
import copy, json, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (BertTokenizer, BertForSequenceClassification, BertForMaskedLM,
                          DataCollatorForLanguageModeling, get_linear_schedule_with_warmup)
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

NAMES = ['Bug Report', 'Feature Request', 'UX Feedback', 'Positive Praise']
MAX_LEN, BATCH_SIZE, LR, MAX_EPOCHS, PATIENCE, SEED = 128, 16, 2e-5, 6, 2, 42

# Baseline from notebook 04, for comparison throughout
BASELINE_ACC, BASELINE_F1 = 0.7212, 0.6754

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')     # Apple Silicon GPU
else:
    device = torch.device('cpu')
print('Device:', device)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps


In [2]:
train_df = pd.read_csv('../data/processed/train.csv')
val_df   = pd.read_csv('../data/processed/val.csv')
test_df  = pd.read_csv('../data/processed/test.csv')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

print(f'Train {len(train_df)} | Val {len(val_df)} | Test {len(test_df)}')
print()
print('Training class distribution (the imbalance this notebook is fighting):')
print(train_df['category_name'].value_counts())

Train 1540 | Val 330 | Test 330

Training class distribution (the imbalance this notebook is fighting):
category_name
Positive Praise    583
Bug Report         520
UX Feedback        307
Feature Request    130
Name: count, dtype: int64


### Shared helpers

Copied from notebook 04 so each experiment changes one thing and nothing else.

In [3]:
def set_seed(s=SEED):
    torch.manual_seed(s)
    np.random.seed(s)


class ReviewDataset(Dataset):
    """Tokenises on access. S2 swaps in a binary target via `label_col`."""
    def __init__(self, texts, labels):
        self.texts, self.labels = list(texts), list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = tokenizer(str(self.texts[i]), max_length=MAX_LEN,
                        padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(),
                'attention_mask': enc['attention_mask'].squeeze(),
                'label': torch.tensor(self.labels[i], dtype=torch.long)}


def make_loader(df, shuffle=False, text_col='clean_text', label_col='category_label'):
    return DataLoader(ReviewDataset(df[text_col], df[label_col]),
                      batch_size=BATCH_SIZE, shuffle=shuffle)


class FocalLoss(nn.Module):
    """Focal loss (Lin et al., ICCV 2017): down-weights easy examples.

    Can be combined with class weights via `weight`; S1 tests whether that helps.
    """
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.weight, self.gamma = weight, gamma

    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, reduction='none')   # unweighted -> true p_t
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        if self.weight is not None:
            focal = self.weight[target] * focal   # class weight as the focal alpha
        return focal.mean()

In [4]:
@torch.no_grad()
def get_logits(model, loader):
    """Raw logits and gold labels. Not argmax, because S0 applies its own
    thresholds and S3 needs probabilities."""
    model.eval()
    all_logits, all_labels = [], []
    for b in loader:
        out = model(input_ids=b['input_ids'].to(device),
                    attention_mask=b['attention_mask'].to(device))
        all_logits.append(out.logits.cpu().numpy())
        all_labels.append(b['label'].numpy())
    return np.concatenate(all_logits), np.concatenate(all_labels)


def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro', zero_division=0)


def report(y_true, y_pred, names=NAMES):
    r = classification_report(y_true, y_pred, target_names=names,
                              output_dict=True, zero_division=0)
    return {'accuracy': round(r['accuracy'], 4),
            'macro_f1': round(r['macro avg']['f1-score'], 4),
            'per_class_f1': {n: round(r[n]['f1-score'], 4) for n in names}}

In [5]:
def finetune(train_data, val_data, tag, num_labels=4, loss_fn=None,
             init='bert-base-uncased', label_col='category_label'):
    """Fine-tune BERT. Identical to notebook 04 unless a parameter overrides it.

    Returns (model, best_val_macro_f1). Early stops on validation macro F1.
    """
    set_seed()
    model = BertForSequenceClassification.from_pretrained(init, num_labels=num_labels).to(device)
    train_loader = make_loader(train_data, shuffle=True, label_col=label_col)
    val_loader = make_loader(val_data, label_col=label_col)

    # Standard BERT practice: no weight decay on bias / LayerNorm parameters
    no_decay = ['bias', 'LayerNorm.weight']
    grouped = [
        {'params': [p for n, p in model.named_parameters() if not any(d in n for d in no_decay)],
         'weight_decay': 0.01},
        {'params': [p for n, p in model.named_parameters() if any(d in n for d in no_decay)],
         'weight_decay': 0.0},
    ]
    optimizer = AdamW(grouped, lr=LR, eps=1e-8)
    total_steps = len(train_loader) * MAX_EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)

    if loss_fn is None:   # default = notebook 04's configuration
        cw = compute_class_weight('balanced', classes=np.arange(num_labels),
                                  y=train_data[label_col].values)
        loss_fn = FocalLoss(torch.tensor(cw, dtype=torch.float).to(device), gamma=2.0)

    best_f1, best_state, patience_left = -1, None, PATIENCE
    for epoch in range(MAX_EPOCHS):
        model.train()
        for b in tqdm(train_loader, desc=f'{tag} ep{epoch+1}', leave=False):
            optimizer.zero_grad()
            out = model(input_ids=b['input_ids'].to(device),
                        attention_mask=b['attention_mask'].to(device))
            loss = loss_fn(out.logits, b['label'].to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        logits, y = get_logits(model, val_loader)
        v = macro_f1(y, logits.argmax(1))
        print(f'  [{tag}] epoch {epoch+1}: val macro F1 = {v:.4f}')

        if v > best_f1:
            best_f1, best_state, patience_left = v, copy.deepcopy(model.state_dict()), PATIENCE
        else:
            patience_left -= 1
            if patience_left == 0:
                print(f'  [{tag}] early stopping at epoch {epoch+1}')
                break

    model.load_state_dict(best_state)
    return model, best_f1


def free(*models):
    """MPS holds onto memory between experiments unless told otherwise."""
    for m in models:
        del m
    if device.type == 'mps':
        torch.mps.empty_cache()

---
## S0: Per-class decision-threshold tuning

If the model ranks classes sensibly but `argmax` is badly calibrated for rare ones, a
per-class logit offset should help without retraining. Offsets are fitted by coordinate
ascent on validation, then applied unchanged to test.

In [6]:
base_model, base_val = finetune(train_df, val_df, 'S0:baseline')

val_logits,  val_y  = get_logits(base_model, make_loader(val_df))
test_logits, test_y = get_logits(base_model, make_loader(test_df))

s0_baseline = report(test_y, test_logits.argmax(1))
print(f'\nBaseline (this run): val macro F1 = {base_val:.4f}')
print(f'Baseline (this run): test = {s0_baseline}')
print(f'Baseline (notebook 04, saved checkpoint): test macro F1 = {BASELINE_F1}')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7640.04it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

  [S0:baseline] epoch 1: val macro F1 = 0.5519


  [S0:baseline] epoch 2: val macro F1 = 0.6135


  [S0:baseline] epoch 3: val macro F1 = 0.6578


  [S0:baseline] epoch 4: val macro F1 = 0.6404


  [S0:baseline] epoch 5: val macro F1 = 0.6727


  [S0:baseline] epoch 6: val macro F1 = 0.6561

Baseline (this run): val macro F1 = 0.6727
Baseline (this run): test = {'accuracy': 0.7061, 'macro_f1': 0.653, 'per_class_f1': {'Bug Report': 0.7137, 'Feature Request': 0.5455, 'UX Feedback': 0.5333, 'Positive Praise': 0.8197}}
Baseline (notebook 04, saved checkpoint): test macro F1 = 0.7079


In [7]:
# Coordinate ascent: sweep one class offset at a time, keep whatever helps on val.
thresholds = np.zeros(4)
best = macro_f1(val_y, (val_logits + thresholds).argmax(1))
print(f'Val macro F1 before tuning: {best:.4f}')

for sweep in range(3):
    for c in range(4):
        for delta in np.arange(-2.0, 2.01, 0.1):
            trial = thresholds.copy()
            trial[c] = delta
            score = macro_f1(val_y, (val_logits + trial).argmax(1))
            if score > best:
                best, thresholds = score, trial
    print(f'  sweep {sweep+1}: val macro F1 = {best:.4f}  offsets = {thresholds.round(2)}')

s0_tuned = report(test_y, (test_logits + thresholds).argmax(1))
print(f'\nTuned:  val = {best:.4f}   test = {s0_tuned}')
print(f'Change: val {best - base_val:+.4f}   test {s0_tuned["macro_f1"] - s0_baseline["macro_f1"]:+.4f}')

Val macro F1 before tuning: 0.6727
  sweep 1: val macro F1 = 0.6797  offsets = [-0.2  0.2 -1.1  0. ]
  sweep 2: val macro F1 = 0.6859  offsets = [-0.1  0.2 -1.1 -0.3]
  sweep 3: val macro F1 = 0.6859  offsets = [-0.1  0.2 -1.1 -0.3]

Tuned:  val = 0.6859   test = {'accuracy': 0.6818, 'macro_f1': 0.6132, 'per_class_f1': {'Bug Report': 0.6797, 'Feature Request': 0.5231, 'UX Feedback': 0.4255, 'Positive Praise': 0.8245}}
Change: val +0.0133   test -0.0398


### S0 result

| | Val | Test |
|---|---|---|
| S0 base model (this run) | 0.6727 | 0.6530 |
| Tuned | **0.6859** | 0.6132 |

Didn't work. Validation up 0.013, test down 0.040. Four free parameters on 330
validation reviews overfits the selection set.

Note this run's own baseline scores 0.6530 on test, not the 0.6754 of notebook 04's
checkpoint, same configuration, different training run. That gap is the subject of S5.

In [8]:
free(base_model)

---
## S1: Loss-function ablation

Notebook 04 applies class weights and focal loss together. Nothing in the literature
recommends both, and Feature Request precision 0.59 against recall 0.70 at the time suggested
over-correction. Four models, differing only in the loss. `ce_plain` is the control.

In [9]:
class_weights = torch.tensor(
    compute_class_weight('balanced', classes=np.arange(4), y=train_df['category_label'].values),
    dtype=torch.float).to(device)
print('Class weights:', class_weights.cpu().numpy().round(3))

loss_variants = {
    'ce_plain':       nn.CrossEntropyLoss(),
    'ce_weighted':    nn.CrossEntropyLoss(weight=class_weights),
    'focal_only':     FocalLoss(None, 2.0),
    'focal_weighted': FocalLoss(class_weights, 2.0),   # notebook 04's setting
}

s1_results = {}
for name, fn in loss_variants.items():
    print(f'\n--- {name} ---')
    m, v = finetune(train_df, val_df, f'S1:{name}', loss_fn=fn)
    lg, y = get_logits(m, make_loader(test_df))
    s1_results[name] = {'val_macro_f1': round(v, 4), **report(y, lg.argmax(1))}
    print(f'  RESULT {name}: val={v:.4f} test_macro_f1={s1_results[name]["macro_f1"]:.4f}')
    free(m)

pd.DataFrame(s1_results).T[['val_macro_f1', 'accuracy', 'macro_f1']]

Class weights: [0.74  2.962 1.254 0.66 ]

--- ce_plain ---


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7226.05it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

  [S1:ce_plain] epoch 1: val macro F1 = 0.4146


  [S1:ce_plain] epoch 2: val macro F1 = 0.5167


  [S1:ce_plain] epoch 3: val macro F1 = 0.6085


...

  [S1:focal_weighted] epoch 2: val macro F1 = 0.5374


  [S1:focal_weighted] epoch 3: val macro F1 = 0.6626


  [S1:focal_weighted] epoch 4: val macro F1 = 0.6462


  [S1:focal_weighted] epoch 5: val macro F1 = 0.6605
  [S1:focal_weighted] early stopping at epoch 5
  RESULT focal_weighted: val=0.6626 test_macro_f1=0.6473


,val_macro_f1,accuracy,macro_f1
ce_plain,0.6337,0.7121,0.6377
ce_weighted,0.6444,0.703,0.6555
focal_only,0.6395,0.7091,0.6472
focal_weighted,0.6626,0.703,0.6473


### S1 result

| Variant | Val | Test |
|---|---|---|
| `ce_plain` | 0.6337 | 0.6377 |
| `ce_weighted` | 0.6444 | **0.6555** |
| `focal_only` | 0.6395 | 0.6472 |
| `focal_weighted` *(nb 04 config)* | **0.6626** | 0.6473 |

Correction is necessary: without it Feature Request falls to 0.39. Plain class-weighted
CE edged focal+weights on test by 0.8 points, which is inside the run-to-run spread S5
measures, so this ablation does not settle the over-correction question on its own.


---
## S2: One-vs-rest binary classifiers

Maalej & Nabil report that binary classifiers beat multiclass on this taxonomy. Four
binary BERT models, each with balanced weights; predict whichever is most confident.

In [10]:
s2_binary = {}
val_probs = np.zeros((len(val_df), 4))
test_probs = np.zeros((len(test_df), 4))

for idx, cname in enumerate(NAMES):
    print(f'\n--- binary: {cname} vs rest ---')
    tr = train_df.copy(); tr['bin'] = (tr['category_label'] == idx).astype(int)
    va = val_df.copy();   va['bin'] = (va['category_label'] == idx).astype(int)
    te = test_df.copy();  te['bin'] = (te['category_label'] == idx).astype(int)

    bw = torch.tensor(compute_class_weight('balanced', classes=np.arange(2), y=tr['bin'].values),
                      dtype=torch.float).to(device)
    m, vf1 = finetune(tr, va, f'S2:{cname[:8]}', num_labels=2,
                      loss_fn=nn.CrossEntropyLoss(weight=bw), label_col='bin')

    vlg, _ = get_logits(m, make_loader(va, label_col='bin'))
    tlg, _ = get_logits(m, make_loader(te, label_col='bin'))
    # probability of the positive class = "this review IS this category"
    val_probs[:, idx]  = torch.softmax(torch.tensor(vlg), dim=1).numpy()[:, 1]
    test_probs[:, idx] = torch.softmax(torch.tensor(tlg), dim=1).numpy()[:, 1]

    s2_binary[cname] = round(vf1, 4)
    print(f'  binary val macro F1 ({cname}): {vf1:.4f}')
    free(m)

s2_val  = macro_f1(val_df['category_label'].values,  val_probs.argmax(1))
s2_test = report(test_df['category_label'].values, test_probs.argmax(1))
print(f'\nS2 COMBINED: val={s2_val:.4f} test={s2_test}')


--- binary: Bug Report vs rest ---


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6831.00it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

  [S2:Bug Repo] epoch 1: val macro F1 = 0.7063


  [S2:Bug Repo] epoch 2: val macro F1 = 0.8311


  [S2:Bug Repo] epoch 3: val macro F1 = 0.8236


...

  [S2:Positive] epoch 1: val macro F1 = 0.8711


  [S2:Positive] epoch 2: val macro F1 = 0.8949


  [S2:Positive] epoch 3: val macro F1 = 0.8979


  [S2:Positive] epoch 4: val macro F1 = 0.8951


  [S2:Positive] epoch 5: val macro F1 = 0.8921
  [S2:Positive] early stopping at epoch 5
  binary val macro F1 (Positive Praise): 0.8979

S2 COMBINED: val=0.7153 test={'accuracy': 0.7061, 'macro_f1': 0.6768, 'per_class_f1': {'Bug Report': 0.6756, 'Feature Request': 0.6087, 'UX Feedback': 0.6324, 'Positive Praise': 0.7905}}


### S2 result

| | Delivered (nb 04) | One-vs-rest |
|---|---|---|
| Test macro F1 | 0.6754 | 0.6768 |
| **Feature Request F1** | 0.58 | **0.61** |
| UX Feedback F1 | 0.59 | 0.63 |

+0.0014 macro F1 overall, a fraction of the run-to-run spread and not an improvement in any
meaningful sense. But Feature Request does rise, which is where Maalej & Nabil predict the
gain, so the mechanism they describe shows up even though the macro average does not move.

Costs four times the training and inference. (`reports/improvement_experiments_notebook_run.json`
holds the exact figures.)

---
## S3: Confident learning, are the labels wrong?

Everything so far assumes my labels are right. Deniz et al. found removing questionable
labels improved accuracy by 8-14 points, which is larger than my whole shortfall.

Confident learning (Northcutt et al., via `cleanlab`): three-fold CV gives out-of-sample
probabilities, cleanlab ranks likely errors, then retrain on the pruned set.

Only the training set is pruned. The test set is audited separately but never changed,
because cleaning the benchmark would invalidate every comparison in the project.

Needs `pip install cleanlab`.

In [11]:
from cleanlab.filter import find_label_issues

y_train = train_df['category_label'].values
oof_probs = np.zeros((len(train_df), 4))    # out-of-sample predictions

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
for k, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y_train)), y_train), 1):
    print(f'\n--- CV fold {k}/3 ---')
    m, _ = finetune(train_df.iloc[tr_idx], train_df.iloc[va_idx], f'S3:cv{k}')
    lg, _ = get_logits(m, make_loader(train_df.iloc[va_idx]))
    oof_probs[va_idx] = torch.softmax(torch.tensor(lg), dim=1).numpy()
    free(m)

print('\nOut-of-sample probabilities computed for all', len(train_df), 'training rows')


--- CV fold 1/3 ---


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5770.97it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

  [S3:cv1] epoch 1: val macro F1 = 0.3894


  [S3:cv1] epoch 2: val macro F1 = 0.5292


  [S3:cv1] epoch 3: val macro F1 = 0.5965


...

  [S3:cv3] epoch 2: val macro F1 = 0.4162


  [S3:cv3] epoch 3: val macro F1 = 0.5595


  [S3:cv3] epoch 4: val macro F1 = 0.5741


  [S3:cv3] epoch 5: val macro F1 = 0.5675


  [S3:cv3] epoch 6: val macro F1 = 0.5689
  [S3:cv3] early stopping at epoch 6

Out-of-sample probabilities computed for all 1540 training rows


In [12]:
issue_idx = find_label_issues(labels=y_train, pred_probs=oof_probs,
                              return_indices_ranked_by='self_confidence')

print(f'cleanlab flagged {len(issue_idx)} / {len(train_df)} training labels '
      f'({100*len(issue_idx)/len(train_df):.1f}%) as likely errors\n')

flagged = train_df.iloc[issue_idx].copy()
flagged['suspect_rank']     = range(1, len(issue_idx) + 1)
flagged['model_would_say']  = [NAMES[i] for i in oof_probs[issue_idx].argmax(1)]
flagged['model_confidence'] = oof_probs[issue_idx].max(1).round(4)

print('Flagged per class:')
print(flagged['category_name'].value_counts())

flagged[['review_id', 'review_text', 'category_name', 'model_would_say',
         'model_confidence', 'suspect_rank']].to_csv(
    '../reports/label_issues_flagged.csv', index=False)
print('\nSaved ../reports/label_issues_flagged.csv  <-- REVIEW THESE BY HAND')

print('\nTop 10 most-suspect labels:')
for _, r in flagged.head(10).iterrows():
    print(f"  you said {r['category_name']:16s} | model says {r['model_would_say']:16s} "
          f"({r['model_confidence']:.2f}) :: {str(r['review_text'])[:80]}")

cleanlab flagged 366 / 1540 training labels (23.8%) as likely errors

Flagged per class:
category_name
Bug Report         142
Positive Praise     81
UX Feedback         81
Feature Request     62
Name: count, dtype: int64

Saved ../reports/label_issues_flagged.csv  <-- REVIEW THESE BY HAND

Top 10 most-suspect labels:
  you said Positive Praise  | model says UX Feedback      (0.51) :: We should be able to hide or delete the "Tasks" section. It's redundant. I think
  you said Positive Praise  | model says UX Feedback      (0.47) :: Changing my review from 5 to 1 - The new update is Bull, you cant even see the b
  you said Positive Praise  | model says UX Feedback      (0.53) :: 6 0ut of 10. Would be a 10 out of 10 if the app allowed MyFitnessPal to read wei
  you said Positive Praise  | model says Bug Report       (0.76) :: website olds rules on my device Samsung galaxy A13 and A175F apps and network re
  you said Feature Request  | model says Bug Report       (0.49) :: System works ok. 

In [13]:
clean_train = train_df.drop(train_df.index[issue_idx]).reset_index(drop=True)
print(f'Pruned training set: {len(clean_train)} rows (was {len(train_df)})')
print(clean_train['category_name'].value_counts(), '\n')

m, v = finetune(clean_train, val_df, 'S3:cleaned')
lg, y = get_logits(m, make_loader(test_df))
s3 = report(y, lg.argmax(1))
print(f'\nS3 RESULT: val={v:.4f} test={s3}')
print(f'vs baseline test macro F1 {BASELINE_F1}: {s3["macro_f1"] - BASELINE_F1:+.4f}')
free(m)

Pruned training set: 1174 rows (was 1540)
category_name
Positive Praise    502
Bug Report         378
UX Feedback        226
Feature Request     68
Name: count, dtype: int64 



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6420.22it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

  [S3:cleaned] epoch 1: val macro F1 = 0.4794


  [S3:cleaned] epoch 2: val macro F1 = 0.6021


  [S3:cleaned] epoch 3: val macro F1 = 0.5920


  [S3:cleaned] epoch 4: val macro F1 = 0.6022


  [S3:cleaned] epoch 5: val macro F1 = 0.6082


  [S3:cleaned] epoch 6: val macro F1 = 0.6244

S3 RESULT: val=0.6244 test={'accuracy': 0.697, 'macro_f1': 0.6362, 'per_class_f1': {'Bug Report': 0.7, 'Feature Request': 0.4583, 'UX Feedback': 0.5816, 'Positive Praise': 0.8048}}
vs baseline test macro F1 0.7079: -0.0717


### S3b - Auditing the test labels

If the benchmark's own labels are noisy, that imposes a hard ceiling on the macro F1
any model can be measured at, however good the model is. This estimates that
ceiling.

Nothing is changed or removed. The output is a percentage and a CSV for manual
inspection, nothing more.

In [14]:
from cleanlab.filter import find_label_issues

audit_model, _ = finetune(train_df, val_df, 'S3:audit')
tlg, ty = get_logits(audit_model, make_loader(test_df))
test_probs_audit = torch.softmax(torch.tensor(tlg), dim=1).numpy()

test_issues = find_label_issues(labels=ty, pred_probs=test_probs_audit,
                                return_indices_ranked_by='self_confidence')
pct = 100 * len(test_issues) / len(test_df)
print(f'cleanlab flags {len(test_issues)}/{len(test_df)} TEST labels ({pct:.1f}%) as suspect')
print('(diagnostic only -- no test label has been changed or removed)\n')

audit = test_df.iloc[test_issues].copy()
audit['model_would_say'] = [NAMES[i] for i in test_probs_audit[test_issues].argmax(1)]
print(audit['category_name'].value_counts())
audit[['review_id', 'review_text', 'category_name', 'model_would_say']].to_csv(
    '../reports/label_issues_test_audit.csv', index=False)
print('\nSaved ../reports/label_issues_test_audit.csv')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6722.94it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

  [S3:audit] epoch 1: val macro F1 = 0.5495


  [S3:audit] epoch 2: val macro F1 = 0.5809


  [S3:audit] epoch 3: val macro F1 = 0.6483


  [S3:audit] epoch 4: val macro F1 = 0.6188


  [S3:audit] epoch 5: val macro F1 = 0.6470
  [S3:audit] early stopping at epoch 5
cleanlab flags 81/330 TEST labels (24.5%) as suspect
(diagnostic only -- no test label has been changed or removed)

category_name
UX Feedback        29
Bug Report         25
Positive Praise    16
Feature Request    11
Name: count, dtype: int64

Saved ../reports/label_issues_test_audit.csv


---
## S4: Task-Adaptive Pretraining

Gururangan et al. show that continuing MLM pretraining on the task's own text helps
downstream classification. No label leakage, since MLM only predicts masked tokens.

Caveat: TAPT normally uses a large unlabelled corpus. Every review I scraped was
labelled, so this runs on ~1,540 short texts. Not expecting much.

In [15]:
set_seed()
mlm_model = BertForMaskedLM.from_pretrained('bert-base-uncased').to(device)

texts = train_df['clean_text'].astype(str).tolist()
enc = tokenizer(texts, max_length=MAX_LEN, padding='max_length',
                truncation=True, return_tensors='pt')
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)


class MLMDataset(Dataset):
    def __len__(self):
        return len(texts)

    def __getitem__(self, i):
        return {'input_ids': enc['input_ids'][i], 'attention_mask': enc['attention_mask'][i]}


mlm_loader = DataLoader(MLMDataset(), batch_size=16, shuffle=True, collate_fn=collator)
mlm_opt = AdamW(mlm_model.parameters(), lr=5e-5)

TAPT_EPOCHS = 5
for ep in range(TAPT_EPOCHS):
    mlm_model.train()
    total = 0.0
    for b in tqdm(mlm_loader, desc=f'TAPT ep{ep+1}', leave=False):
        mlm_opt.zero_grad()
        out = mlm_model(input_ids=b['input_ids'].to(device),
                        attention_mask=b['attention_mask'].to(device),
                        labels=b['labels'].to(device))
        out.loss.backward()
        mlm_opt.step()
        total += out.loss.item()
    print(f'  TAPT epoch {ep+1}: mlm_loss = {total/len(mlm_loader):.4f}')

TAPT_DIR = '../models/bert_tapt'
mlm_model.save_pretrained(TAPT_DIR)
tokenizer.save_pretrained(TAPT_DIR)
free(mlm_model)
print(f'\nTAPT checkpoint saved to {TAPT_DIR}')

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 6649.32it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  TAPT epoch 1: mlm_loss = 3.2146


  TAPT epoch 2: mlm_loss = 3.0113


  TAPT epoch 3: mlm_loss = 2.7182


  TAPT epoch 4: mlm_loss = 2.6341


  TAPT epoch 5: mlm_loss = 2.5053


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.35it/s]


TAPT checkpoint saved to ../models/bert_tapt


In [16]:
m, v = finetune(train_df, val_df, 'S4:tapt', init=TAPT_DIR)
lg, y = get_logits(m, make_loader(test_df))
s4 = report(y, lg.argmax(1))
print(f'\nS4 RESULT: val={v:.4f} test={s4}')
print(f'vs baseline test macro F1 {BASELINE_F1}: {s4["macro_f1"] - BASELINE_F1:+.4f}')
free(m)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 6997.20it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: ../models/bert_tapt
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkp

  [S4:tapt] epoch 1: val macro F1 = 0.6459


  [S4:tapt] epoch 2: val macro F1 = 0.6398


  [S4:tapt] epoch 3: val macro F1 = 0.6406
  [S4:tapt] early stopping at epoch 3

S4 RESULT: val=0.6459 test={'accuracy': 0.7, 'macro_f1': 0.6368, 'per_class_f1': {'Bug Report': 0.6992, 'Feature Request': 0.4583, 'UX Feedback': 0.5606, 'Positive Praise': 0.8291}}
vs baseline test macro F1 0.7079: -0.0711


---
## S5: Seed ensembling, and a caveat that became part of the finding

This targets the problem the other experiments uncovered rather than a modelling
weakness. S0-S4 scored between 0.61 and 0.68, and the delivered configuration retrained here
(S1d) scored 0.6473 against notebook 04's 0.6754 for the same recipe. If that variance is the real obstacle, the answer isn't to keep resampling
until a good run appears, it's to average over it.

The loop below sets `SEEDS = [42, 1, 7, 123, 2024]` intending five different seeds, but
they do **not** take effect: `set_seed(s=SEED)` binds its default to `SEED` at definition
time, and `finetune()` calls `set_seed()` with no argument, so every run uses seed 42
regardless of the loop. All five runs therefore share one seed, and what varies between
them is only what a fixed seed does not control, the unseeded MPS reductions, dataloader
ordering and kernel selection of the environment. This is best read as
**fixed-seed run-to-run variance**, which is a sharper reproducibility result than seed
sensitivity: the numbers move even with the seed held constant. The softmax-averaged
ensemble is still a valid fixed procedure; only the label on the spread changes.

Picking the best of five runs would be selecting on noise. Averaging is a fixed
procedure anyone can repeat, and printing the individual scores makes the spread, and the
standard deviation my single-split results lack, visible.

In [17]:
SEEDS = [42, 1, 7, 123, 2024]
seed_probs, seed_val, seed_test = [], [], []

for s in SEEDS:
    print(f'\n--- seed {s} ---')
    globals()['SEED'] = s          # no effect: set_seed() uses its frozen default (42); these are fixed-seed repeat runs (see note above)
    m, v = finetune(train_df, val_df, f'S5:seed{s}')
    tlg, ty = get_logits(m, make_loader(test_df))
    seed_probs.append(torch.softmax(torch.tensor(tlg), dim=1).numpy())
    one = report(ty, tlg.argmax(1))
    seed_val.append(round(v, 4)); seed_test.append(one['macro_f1'])
    print(f"  seed {s}: val={v:.4f} test={one['macro_f1']:.4f}")
    free(m)

globals()['SEED'] = 42


--- seed 42 ---


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6848.88it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

  [S5:seed42] epoch 1: val macro F1 = 0.5379


  [S5:seed42] epoch 2: val macro F1 = 0.5783


  [S5:seed42] epoch 3: val macro F1 = 0.6440


...

  [S5:seed2024] epoch 1: val macro F1 = 0.5379


  [S5:seed2024] epoch 2: val macro F1 = 0.5855


  [S5:seed2024] epoch 3: val macro F1 = 0.6492


  [S5:seed2024] epoch 4: val macro F1 = 0.6114


  [S5:seed2024] epoch 5: val macro F1 = 0.6457
  [S5:seed2024] early stopping at epoch 5
  seed 2024: val=0.6492 test=0.6373


In [18]:
arr = np.array(seed_test)
print('per-seed test macro F1:', seed_test)
print(f'  mean {arr.mean():.4f}  std {arr.std():.4f}  spread {arr.max()-arr.min():.4f}')
print(f'  (gap to the 0.75 target is {0.75-BASELINE_F1:.4f})')

s5 = report(test_df['category_label'].values, np.mean(seed_probs, axis=0).argmax(1))
print(f"\nensemble of {len(SEEDS)} seeds: {s5}")
print(f"  vs mean single seed: {s5['macro_f1']-arr.mean():+.4f}")
print(f"  vs nb04 baseline:    {s5['macro_f1']-BASELINE_F1:+.4f}")

per-seed test macro F1: [0.5986, 0.6507, 0.6317, 0.6364, 0.6373]
  mean 0.6309  std 0.0174  spread 0.0521
  (gap to the 0.75 target is 0.0421)

ensemble of 5 seeds: {'accuracy': 0.697, 'macro_f1': 0.6383, 'per_class_f1': {'Bug Report': 0.6979, 'Feature Request': 0.5085, 'UX Feedback': 0.521, 'Positive Praise': 0.8259}}
  vs mean single seed: +0.0074
  vs nb04 baseline:    -0.0696


---
## S6: Combining what worked

Two directions looked promising: one-vs-rest (S2) and, in the ablation, plain
class-weighted cross-entropy. S2's binary heads already use weighted CE, so the open
question is whether ensembling each head over seeds compounds the gain.

Four binary heads, three seeds each, positive-class probabilities averaged before the
final argmax. Twelve trainings, so this is the slowest cell here, around 45 minutes.

If this does reach 0.75 it has to be reported as the output of a specified procedure,
not as the baseline model's score. The delivered checkpoint stays at 0.6754.

In [19]:
ENSEMBLE_SEEDS = [42, 1, 7]
s6_probs = np.zeros((len(test_df), 4))
s6_val = {}

for idx, cname in enumerate(NAMES):
    print(f'\n=== {cname} vs rest ===')
    tr = train_df.copy(); tr['bin'] = (tr['category_label'] == idx).astype(int)
    va = val_df.copy();   va['bin'] = (va['category_label'] == idx).astype(int)
    te = test_df.copy();  te['bin'] = (te['category_label'] == idx).astype(int)
    bw = torch.tensor(compute_class_weight('balanced', classes=np.arange(2), y=tr['bin'].values),
                      dtype=torch.float).to(device)

    head, vals = [], []
    for s in ENSEMBLE_SEEDS:
        globals()['SEED'] = s
        m, vf1 = finetune(tr, va, f'S6:{cname[:6]}s{s}', num_labels=2,
                          loss_fn=nn.CrossEntropyLoss(weight=bw), label_col='bin')
        tlg, _ = get_logits(m, make_loader(te, label_col='bin'))
        head.append(torch.softmax(torch.tensor(tlg), dim=1).numpy()[:, 1])
        vals.append(round(vf1, 4)); free(m)

    globals()['SEED'] = 42
    s6_probs[:, idx] = np.mean(head, axis=0)
    s6_val[cname] = vals
    print(f'  per-seed val macro F1: {vals}')

s6 = report(test_df['category_label'].values, s6_probs.argmax(1))
print(f'\none-vs-rest + seed ensemble: {s6}')
print(f"  vs S2 alone (0.7274): {s6['macro_f1']-0.7274:+.4f}")
print(f"  vs nb04 baseline:     {s6['macro_f1']-BASELINE_F1:+.4f}")
print(f"  vs 0.75 target:       {s6['macro_f1']-0.75:+.4f}")


=== Bug Report vs rest ===


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7587.33it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

  [S6:Bug Res42] epoch 1: val macro F1 = 0.7006


  [S6:Bug Res42] epoch 2: val macro F1 = 0.8280


  [S6:Bug Res42] epoch 3: val macro F1 = 0.8285


...

  [S6:Positis7] epoch 2: val macro F1 = 0.8894


  [S6:Positis7] epoch 3: val macro F1 = 0.8936


  [S6:Positis7] epoch 4: val macro F1 = 0.8982


  [S6:Positis7] epoch 5: val macro F1 = 0.9015


  [S6:Positis7] epoch 6: val macro F1 = 0.9009
  per-seed val macro F1: [0.8946, 0.8966, 0.9015]

one-vs-rest + seed ensemble: {'accuracy': 0.7394, 'macro_f1': 0.7124, 'per_class_f1': {'Bug Report': 0.7162, 'Feature Request': 0.6792, 'UX Feedback': 0.624, 'Positive Praise': 0.83}}
  vs S2 alone (0.7274): -0.0150
  vs nb04 baseline:     +0.0045
  vs 0.75 target:       -0.0376


In [20]:
# Only records experiments that actually ran in this session, so the cell works
# whether you ran everything or just part of it.
summary = {
    '_method': ('Selected on VALIDATION. Test figures reported for transparency but never '
                'used for selection. Baseline = notebook 04 '
                f'(test acc {BASELINE_ACC}, macro F1 {BASELINE_F1}).'),
    '_environment': 'notebook kernel run',
}

if 's0_tuned' in dir():
    summary['S0_threshold_tuned'] = {'val_macro_f1': round(best, 4), **s0_tuned}
if 's1_results' in dir():
    summary['S1_loss_ablation'] = s1_results
if 's2_test' in dir():
    summary['S2_one_vs_rest'] = {'per_class_binary_val': s2_binary,
                                 'combined': {'val_macro_f1': round(s2_val, 4), **s2_test}}
if 's3' in dir():
    summary['S3_confident_learning'] = {
        'paper': 'Northcutt et al., JAIR 2021 (arXiv:1911.00068), via cleanlab',
        'n_flagged': int(len(issue_idx)),
        'pct_flagged': round(100 * len(issue_idx) / len(train_df), 2),
        'clean_train_rows': len(clean_train), **s3}
if 'test_issues' in dir():
    summary['S3b_test_label_audit'] = {
        'n_flagged': int(len(test_issues)), 'pct_flagged': round(pct, 2),
        'note': 'DIAGNOSTIC ONLY -- no test label was changed or removed.'}
if 's4' in dir():
    summary['S4_tapt'] = {'paper': 'Gururangan et al., ACL 2020 (arXiv:2004.10964)',
                          'tapt_epochs': TAPT_EPOCHS, **s4}
if 's5' in dir():
    summary['S5_seed_ensemble'] = {
        'paper': 'Lakshminarayanan et al., NeurIPS 2017',
        'seeds': SEEDS, 'individual_test_macro_f1': seed_test,
        'mean': round(float(np.mean(seed_test)), 4),
        'std': round(float(np.std(seed_test)), 4), 'ensemble': s5}
if 's6' in dir():
    summary['S6_ovr_plus_ensemble'] = {'seeds': ENSEMBLE_SEEDS,
                                       'per_head_val': s6_val, 'ensemble': s6}

# Separate filename on purpose: improvement_experiments.json holds the earlier
# standalone-script run, and both are needed for the two-run comparison.
out = '../reports/improvement_experiments_notebook_run.json'
with open(out, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'saved {out} with {len(summary) - 2} experiments')

rows = [('baseline (notebook 04)', None, BASELINE_F1)]
if 's0_tuned' in dir(): rows.append(('S0 threshold-tuned', round(best, 4), s0_tuned['macro_f1']))
if 's1_results' in dir(): rows += [(f'S1 {k}', v['val_macro_f1'], v['macro_f1']) for k, v in s1_results.items()]
if 's2_test' in dir(): rows.append(('S2 one-vs-rest', round(s2_val, 4), s2_test['macro_f1']))
if 's3' in dir(): rows.append(('S3 label-cleaned', None, s3['macro_f1']))
if 's4' in dir(): rows.append(('S4 TAPT', None, s4['macro_f1']))
if 's5' in dir(): rows.append((f'S5 seed ensemble (n={len(SEEDS)})', None, s5['macro_f1']))
if 's6' in dir(): rows.append(('S6 one-vs-rest + ensemble', None, s6['macro_f1']))

pd.DataFrame(rows, columns=['experiment', 'val_macro_f1', 'test_macro_f1'])

saved ../reports/improvement_experiments_notebook_run.json with 8 experiments


,experiment,val_macro_f1,test_macro_f1
0,baseline (notebook 04),NaN,0.7079
1,S0 threshold-tuned,0.6859,0.6132
2,S1 ce_plain,0.6337,0.6377
3,S1 ce_weighted,0.6444,0.6555
4,S1 focal_only,0.6395,0.6472
5,S1 focal_weighted,0.6626,0.6473
6,S2 one-vs-rest,0.7153,0.6768
7,S3 label-cleaned,NaN,0.6362
8,S4 TAPT,NaN,0.6368
9,S5 seed ensemble (n=5),NaN,0.6383
